# 13 Fairness vs Accuracy Tradeoff\n\n## Objectives\n1. Compare model performance\n2. Summarize SHAP insights\n3. Analyze bias metrics\n4. Compare mitigation methods\n5. Discuss business implications\n6. Conclude on ethical AI tradeoffs


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd  # noqa: E402

In [ ]:
import json
from pathlib import Path

base = (
    Path("../reports/fairness_reports")
    if Path.cwd().name == "notebooks"
    else Path("reports/fairness_reports")
)


def read_json(name):
    p = base / name
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    return None


log = read_json("logistic_metrics.json")
xgb = read_json("xgboost_metrics.json")
mit = read_json("xgboost_model_bias_mitigation_summary.json") or read_json(
    "logistic_model_bias_mitigation_summary.json"
)

perf_table = pd.DataFrame(
    [
        {"model": "Logistic", **(log["metrics"] if log else {})},
        {"model": "XGBoost", **(xgb["metrics"] if xgb else {})},
    ]
)
perf_table

In [ ]:
if mit:
    rows = []
    for method in ["baseline", "reweighing", "fairlearn_postprocessing"]:
        if method in mit:
            row = {"method": method}
            row.update({f"perf_{k}": v for k, v in mit[method]["performance"].items()})
            row.update({f"fair_{k}": v for k, v in mit[method]["fairness"].items()})
            rows.append(row)
    tradeoff_df = pd.DataFrame(rows)
    display(tradeoff_df)
else:
    print("Run src.bias_mitigation first.")

## Business Implications\n- Use high-AUC model for screening, with fairness constraints for decision thresholding.\n- Track fairness metrics in governance reports.\n\n## Ethical AI Conclusion\nA strong credit model can remain useful and transparent, but fairness constraints typically require measurable performance tradeoffs that should be explicitly governed.
